In [11]:
import pandas as pd

In [22]:
coin = 'btc'
staking = False
chain = True
retarded = False
#########################################
# Step 1. Load Data
#########################################
avax_prices    = pd.read_parquet(f'../data/price_data/processed/{coin}_processed_data.parquet')
if chain:
    if retarded:
        avax1_features = pd.read_parquet(f'../data/on_chain_data/processed/{coin}1_chain_processed.parquet').drop(columns=['average_difficulty'])
        avax2_features = pd.read_parquet(f'../data/on_chain_data/processed/{coin}2_chain_processed.parquet').drop(columns=['average_base_fee_per_gas','average_difficulty'])
    else:
        avax1_features = pd.read_parquet(f'../data/on_chain_data/processed/{coin}_chain_processed.parquet').drop(columns=['average_difficulty'])
if staking:
    avax_staking   = pd.read_parquet(f'../data/on_chain_data/processed/{coin}_staking_data.parquet')
avax_tech      = pd.read_parquet(f'../data/data_segmented_tech/{coin}_data.parquet')

#########################################
# Step 2. Convert Indices to DateTime
#########################################
if staking:
    avax_staking.index   = pd.to_datetime(avax_staking.index)

avax1_features.index = pd.to_datetime(avax1_features.index)
if retarded:
    avax2_features.index = pd.to_datetime(avax2_features.index)

avax_prices['time_close'] = pd.to_datetime(avax_prices['time_close'])
avax_prices['time_close'] = avax_prices['time_close'].dt.tz_localize('UTC')
avax_prices.set_index('time_close', inplace=True)
avax_prices.index = avax_prices.index + pd.Timedelta(minutes=1)

avax_tech['time_close'] = pd.to_datetime(avax_tech['time_close'])
avax_tech['time_close'] = avax_tech['time_close'].dt.round('min')
avax_tech.set_index('time_close', inplace=True)

#########################################
# Step 3. Combine On-Chain Features
#########################################
if retarded:
    df_combined = pd.concat([avax1_features, avax2_features]).sort_index()
else:
    df_combined = avax1_features.sort_index()

#########################################
# Step 4. Align Staking Data with Combined Features
#########################################
if staking:
    common_start = max(avax_staking.index.min(), df_combined.index.min())
    common_end   = min(avax_staking.index.max(), df_combined.index.max())

if staking:
    df1_aligned         = avax_staking.loc[common_start:common_end]
df_combined_aligned = df_combined.loc[common_start:common_end]

avax_features = pd.merge(
    df1_aligned, 
    df_combined_aligned, 
    left_index=True, 
    right_index=True, 
    how='inner'
)

#########################################
# Step 6. Clean and Align Price Data
#########################################
# Determine the global common time range across features, prices, and tech data
global_start = max(avax_features.index.min(), avax_prices.index.min(), avax_tech.index.min())
global_end   = min(avax_features.index.max(), avax_prices.index.max(), avax_tech.index.max())

avax_features_aligned = avax_features.loc[global_start:global_end]
avax_prices_aligned   = avax_prices.loc[global_start:global_end]

#########################################
# Step 7. Process and Align Technical Indicator Data
#########################################
# Subset tech data using the same global common time range
avax_tech_aligned = avax_tech.loc[global_start:global_end].copy()

# Ensure both DataFrames have unique indices by dropping duplicates
avax_prices_aligned = avax_prices_aligned[~avax_prices_aligned.index.duplicated(keep='first')]
avax_tech_aligned = avax_tech_aligned[~avax_tech_aligned.index.duplicated(keep='first')]

# Reindex the tech data to match the unique price data timestamps using nearest match
avax_tech_aligned = avax_tech_aligned.reindex(avax_prices_aligned.index, method='nearest')

tech_cols = ['fib_23', 'fib_38', 'fib_50', 'fib_61', 'fib_78', 'bollinger', 'EMAcross', 'RSI']
avax_tech_selected = avax_tech_aligned[tech_cols].rename(columns=lambda x: "tech_" + x)

#########################################
# Step 8. Join Price Data with Feature Data
#########################################

data = avax_features_aligned.join(
    avax_prices_aligned[['price_close']], 
    how='inner'
)

#########################################
# Step 9. Join Tech Features to Main Data
#########################################

data = data.join(avax_tech_selected, how='left')

#########################################
# Step 10. Unified Lag Adjustment for All Feature Columns
#########################################

lag_cols = [
    'average_size', 
    'active_validators', 'real_reward_rate',
    'staked_tokens', 'staking_ratio', 'total_staking_wallets'
]
tech_cols_renamed = ["tech_" + col for col in tech_cols]
all_lag_cols = lag_cols + tech_cols_renamed

# Apply a lag of 1 to all feature columns (price_close remains unshifted)
print("Missing values per column:\n", data.isna().sum())
data[all_lag_cols] = data[all_lag_cols].shift(1)
print("Missing values per column:\n", data.isna().sum())

outpath = f'../Data/do_not_load/{coin}_data.parquet'
data.to_parquet(outpath, engine="pyarrow", index=False)

Missing values per column:
 active_validators            0
real_reward_rate             0
staked_tokens                0
staking_ratio                0
total_staking_wallets        0
average_height               0
average_total_fees           0
average_total_reward         0
average_mint_reward          0
average_transaction_count    0
average_nonce                0
average_size                 0
average_stripped_size        0
average_version              0
average_weight               0
price_close                  0
tech_fib_23                  0
tech_fib_38                  0
tech_fib_50                  0
tech_fib_61                  0
tech_fib_78                  0
tech_bollinger               0
tech_EMAcross                0
tech_RSI                     0
dtype: int64
Missing values per column:
 active_validators            1
real_reward_rate             1
staked_tokens                1
staking_ratio                1
total_staking_wallets        1
average_height               0
a

In [ ]:
#########################################
# Step 1. Load Data
#########################################
coin = 'arb'
avax_prices    = pd.read_parquet(f'../data/price_data/processed/{coin}_processed_data.parquet')
avax1_features = pd.read_parquet(f'../data/on_chain_data/processed/{coin}1_chain_processed.parquet').drop(columns=['average_difficulty'])
avax2_features = pd.read_parquet(f'../data/on_chain_data/processed/{coin}2_chain_processed.parquet').drop(columns=['average_base_fee_per_gas','average_difficulty'])
avax_staking   = pd.read_parquet(f'../data/on_chain_data/processed/{coin}_staking_data2.parquet')
avax_tech      = pd.read_parquet(f'../data/data_segmented_tech/{coin}_data.parquet')

#########################################
# Step 2. Convert Indices to DateTime
#########################################
avax_staking.index   = pd.to_datetime(avax_staking.index)
avax1_features.index = pd.to_datetime(avax1_features.index)
avax2_features.index = pd.to_datetime(avax2_features.index)

avax_prices['time_close'] = pd.to_datetime(avax_prices['time_close'])
avax_prices['time_close'] = avax_prices['time_close'].dt.tz_localize('UTC')
avax_prices.set_index('time_close', inplace=True)
avax_prices.index = avax_prices.index + pd.Timedelta(minutes=1)

avax_tech['time_close'] = pd.to_datetime(avax_tech['time_close'])
avax_tech['time_close'] = avax_tech['time_close'].dt.round('min')
avax_tech.set_index('time_close', inplace=True)

#########################################
# Step 3. Combine On-Chain Features
#########################################
df_combined = pd.concat([avax1_features, avax2_features]).sort_index()

#########################################
# Step 4. Align Staking Data with Combined Features
#########################################
common_start = max(avax_staking.index.min(), df_combined.index.min())
common_end   = min(avax_staking.index.max(), df_combined.index.max())

df1_aligned         = avax_staking.loc[common_start:common_end]
df_combined_aligned = df_combined.loc[common_start:common_end]

avax_features = pd.merge(
    df1_aligned, 
    df_combined_aligned, 
    left_index=True, 
    right_index=True, 
    how='inner'
)

#########################################
# Step 6. Clean and Align Price Data
#########################################
# Determine the global common time range across features, prices, and tech data
global_start = max(avax_features.index.min(), avax_prices.index.min(), avax_tech.index.min())
global_end   = min(avax_features.index.max(), avax_prices.index.max(), avax_tech.index.max())

avax_features_aligned = avax_features.loc[global_start:global_end]
avax_prices_aligned   = avax_prices.loc[global_start:global_end]

#########################################
# Step 7. Process and Align Technical Indicator Data
#########################################
# Subset tech data using the same global common time range
avax_tech_aligned = avax_tech.loc[global_start:global_end].copy()

# Ensure both DataFrames have unique indices by dropping duplicates
avax_prices_aligned = avax_prices_aligned[~avax_prices_aligned.index.duplicated(keep='first')]
avax_tech_aligned = avax_tech_aligned[~avax_tech_aligned.index.duplicated(keep='first')]

# Reindex the tech data to match the unique price data timestamps using nearest match
avax_tech_aligned = avax_tech_aligned.reindex(avax_prices_aligned.index, method='nearest')

tech_cols = ['fib_23', 'fib_38', 'fib_50', 'fib_61', 'fib_78', 'bollinger', 'EMAcross', 'RSI']
avax_tech_selected = avax_tech_aligned[tech_cols].rename(columns=lambda x: "tech_" + x)

#########################################
# Step 8. Join Price Data with Feature Data
#########################################

data = avax_features_aligned.join(
    avax_prices_aligned[['price_close']], 
    how='inner'
)

#########################################
# Step 9. Join Tech Features to Main Data
#########################################

data = data.join(avax_tech_selected, how='left')

#########################################
# Step 10. Unified Lag Adjustment for All Feature Columns
#########################################

lag_cols = [
    'average_gas_limit', 'average_gas_used', 'average_size', 
    'average_total_difficulty', 'active_validators', 'real_reward_rate',
    'staked_tokens', 'staking_ratio', 'total_staking_wallets'
]
tech_cols_renamed = ["tech_" + col for col in tech_cols]
all_lag_cols = lag_cols + tech_cols_renamed

# Apply a lag of 1 to all feature columns (price_close remains unshifted)
print("Missing values per column:\n", data.isna().sum())
data[all_lag_cols] = data[all_lag_cols].shift(1)
print("Missing values per column:\n", data.isna().sum())

KeyError: "['average_difficulty'] not found in axis"